# 10 — CFD Validation (Module 7 — Verification Loop)

Runs OpenFOAM `reactingFoam` CFD on the **top-3 designs** from `geometry_loop_results.json`,
then compares surrogate predictions against the CFD ground truth.

**Two-phase workflow**
| Phase | Where | Cells |
|-------|-------|-------|
| Generate cases | Colab | 0 – 2 |
| Run CFD | Machine with Docker | Cell 3 (commands) or Cell 4 (programmatic) |
| Postprocess + compare | Colab | 5 – 7 |

In [ ]:
# ── Cell 0: Setup ──────────────────────────────────────────────────────────
import subprocess, sys
subprocess.run(['git', '-C', '/content/cfd-ald-app', 'pull'], check=False)
sys.path.insert(0, '/content/cfd-ald-app')

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# Install deps not in base Colab image
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'h5py', 'torch_geometric', 'trimesh'], check=False)

import json, shutil
import numpy as np
import h5py
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.spatial import cKDTree

DRIVE_BASE  = Path('/content/drive/MyDrive/cfd-ald-app')
GL_JSON     = DRIVE_BASE / 'checkpoints' / 'geometry_loop' / 'geometry_loop_results.json'
CFD_CASES   = DRIVE_BASE / 'checkpoints' / 'cfd_validation' / 'cases'
HDF5_OUT    = DRIVE_BASE / 'checkpoints' / 'cfd_validation' / 'hdf5'
VAL_JSON    = DRIVE_BASE / 'checkpoints' / 'cfd_validation' / 'validation_results.json'
CKPT        = DRIVE_BASE / 'checkpoints' / 'multihead' / 'multihead_final.pt'
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

CFD_CASES.mkdir(parents=True, exist_ok=True)
HDF5_OUT.mkdir(parents=True, exist_ok=True)

# Shared geometry base params (same as param_sweep.py GEOM_BASE)
GEOM_BASE = {
    'H_plenum':  0.020,   # 20 mm plenum height
    't_face':    0.003,   # 3 mm faceplate thickness
    'standoff':  0.020,   # 20 mm nozzle-exit to wafer distance
    'D_plate':   0.300,   # 300 mm plate diameter
    'theta_deg': 0.0,
}

print(f'Setup complete. Device: {DEVICE}')
print(f'Cases dir : {CFD_CASES}')
print(f'HDF5  dir : {HDF5_OUT}')

In [ ]:
# ── Cell 1: Load top-3 designs ─────────────────────────────────────────────
with open(GL_JSON) as f:
    gl = json.load(f)

top3 = [d for d in gl['ranked_designs'] if d.get('flag_for_cfd')]
assert top3, 'No designs flagged for CFD in geometry_loop_results.json'

print(f'Loaded {len(top3)} designs flagged for CFD validation:\n')
for d in top3:
    print(f"  Rank {d['rank']}: D={d['D_mm']:.2f}mm  pitch/D={d['pitch_over_D']:.1f}  "
          f"Q={d['Q_slm']:.2f}slm  n_holes={d['n_holes']}  score={d['score']:.4f}")

In [ ]:
# ── Cell 2: Generate OpenFOAM case directories ─────────────────────────────
from geometry.grammar import ShowerheadTopology, NozzlePattern
from geometry.parametric import build_showerhead
from geometry.quality_check import check_geometry
from openfoam.case_generator import generate_case

generated_cases = []   # list of Path objects

for d in top3:
    geo_params = dict(GEOM_BASE)
    geo_params['D']            = d['D_mm'] / 1000.0   # mm → m
    geo_params['pitch_over_D'] = d['pitch_over_D']

    topology = ShowerheadTopology(pattern=NozzlePattern.HEX)
    geo      = build_showerhead(geo_params, topology)

    # Geometry quality check
    report = check_geometry(geo)
    if not report.passed:
        errs = [i.message for i in report.issues if i.level == 'error']
        print(f'  [SKIP] Rank {d["rank"]}: geometry quality failed — {", ".join(errs)}')
        continue

    case_name = (f"cfd_val_rank{d['rank']}_D{d['D_mm']:.1f}mm"
                 f"_p{d['pitch_over_D']:.1f}_Q{d['Q_slm']:.2f}slm")
    case_dir  = CFD_CASES / case_name

    if case_dir.exists():
        print(f'  [SKIP] {case_name}: directory already exists')
    else:
        generate_case(
            geo=geo,
            case_dir=str(case_dir),
            flow_rate_slm=d['Q_slm'],
            beta=0.05,
            v_th=144.0,
            D_m=2.5e-5,
            pulse_time=0.10,
            purge_time=0.10,
            dt=1e-4,
        )
        print(f'  [GEN ] {case_name}  ({geo.n_holes} holes)')

    generated_cases.append(case_dir)

print(f'\n{len(generated_cases)} case directories ready.')

In [ ]:
# ── Cell 3: Docker run commands ────────────────────────────────────────────
# Colab does NOT have Docker. Run these commands on a machine with Docker
# (local workstation / cloud VM) where Google Drive is also mounted.
#
# Each case: ~2-4 hours depending on mesh density (D=1mm, ~3000 holes).
# Run all three sequentially or in parallel (if you have enough RAM/cores).

print('=' * 70)
print('DOCKER COMMANDS — run these on a machine with Docker + Drive mounted')
print('=' * 70)

script_lines = ['#!/bin/bash', 'set -e', '']
for case_dir in generated_cases:
    cmd = (f'docker run --rm \\\n'
           f'    -v "{case_dir}":/case \\\n'
           f'    opencfd/openfoam-default:latest \\\n'
           f'    /bin/bash /case/run.sh')
    print(f'\n# {case_dir.name}')
    print(cmd)
    script_lines += [f'# {case_dir.name}', cmd, '']

# Save a run_all.sh helper script to Drive
run_all = CFD_CASES.parent / 'run_all_validation.sh'
run_all.write_text('\n'.join(script_lines))
print(f'\nSaved helper script → {run_all}')
print('\nThen run Cell 5 (postprocess) once Docker finishes.')

In [ ]:
# ── Cell 4: Run Docker programmatically (skip if running on Colab) ─────────
# Run this cell only on a machine with Docker available.
# On Colab: skip this cell and use the commands from Cell 3 instead.
import shutil as _sh

if _sh.which('docker') is None:
    print('Docker not found — skip this cell and use Cell 3 commands on a local machine.')
else:
    from openfoam.param_sweep import run_case_docker
    for case_dir in generated_cases:
        print(f'\nRunning {case_dir.name} …')
        result = run_case_docker(case_dir, verbose=True)
        print(f'  status: {result["status"]}')
        if result.get('error'):
            print(f'  error : {result["error"]}')

In [ ]:
# ── Cell 5: Postprocess completed cases to HDF5 ────────────────────────────
# Run AFTER Docker CFD runs complete.
# Checks each case has time dirs > 0 with U field, then calls process_case().
from openfoam.postprocess import process_case

def _is_completed(case_dir: Path) -> bool:
    for d in case_dir.iterdir():
        if d.is_dir() and d.name not in ('0', 'constant', 'system', 'geometry'):
            try:
                t = float(d.name)
                if t > 0 and (d / 'U').exists():
                    return True
            except ValueError:
                pass
    return False

postprocessed = []   # list of (case_dir, h5_path)

for case_dir in generated_cases:
    if not _is_completed(case_dir):
        print(f'  [WAIT] {case_dir.name}: CFD not yet complete — run Docker first')
        continue

    h5_stem = case_dir.name + '.h5'
    h5_path = HDF5_OUT / h5_stem

    if h5_path.exists():
        print(f'  [SKIP] {h5_stem}: already postprocessed')
    else:
        print(f'  [POST] {case_dir.name} …', end=' ', flush=True)
        result = process_case(case_dir, HDF5_OUT)
        print(f'done  N={result.get("n_points", "?")} pts  →  {h5_stem}')

    postprocessed.append((case_dir, h5_path))

print(f'\n{len(postprocessed)} / {len(generated_cases)} cases postprocessed.')

In [ ]:
# ── Cell 6: Load MultiHeadMGN surrogate ────────────────────────────────────
GLOBAL_KEYS = [
    'Re', 'Pr', 'Sc', 'Ma', 'Pe_h', 'Pe_m', 'Da',
    'D', 'pitch_over_D', 'H_plenum', 't_face', 'standoff',
    'flow_rate_slm', 'beta', 'v_th', 'D_m', 'n_holes', 'open_area_frac',
]

def mlp(in_dim, out_dim, hidden=None, n_layers=2):
    hidden = hidden or out_dim
    dims   = [in_dim] + [hidden] * (n_layers - 1) + [out_dim]
    mods   = []
    for i in range(len(dims) - 1):
        mods.append(nn.Linear(dims[i], dims[i+1]))
        if i < len(dims) - 2:
            mods.append(nn.SiLU())
    return nn.Sequential(*mods)

from torch_geometric.nn import MessagePassing

class MGNProcessor(MessagePassing):
    def __init__(self, hidden):
        super().__init__(aggr='sum')
        self.edge_mlp  = mlp(3*hidden, hidden)
        self.node_mlp  = mlp(2*hidden, hidden)
        self.edge_norm = nn.LayerNorm(hidden)
        self.node_norm = nn.LayerNorm(hidden)
    def forward(self, x, edge_index, edge_attr):
        src, dst = edge_index
        new_e = self.edge_norm(self.edge_mlp(torch.cat([x[src], x[dst], edge_attr], -1)) + edge_attr)
        agg   = self.propagate(edge_index, x=x, edge_attr=new_e, size=(x.size(0), x.size(0)))
        new_x = self.node_norm(self.node_mlp(torch.cat([x, agg], -1)) + x)
        return new_x, new_e
    def message(self, edge_attr): return edge_attr

class MultiHeadMGN(nn.Module):
    def __init__(self, node_dim, edge_dim, flow_out=4, heat_out=1,
                 species_out=1, hidden=128, n_layers=8):
        super().__init__()
        self.node_enc    = mlp(node_dim, hidden)
        self.edge_enc    = mlp(edge_dim, hidden)
        self.processors  = nn.ModuleList([MGNProcessor(hidden) for _ in range(n_layers)])
        self.flow_dec    = mlp(hidden, flow_out)
        self.heat_dec    = mlp(hidden, heat_out)
        self.species_dec = mlp(hidden, species_out)
    def forward(self, x, edge_index, edge_attr):
        x = self.node_enc(x)
        e = self.edge_enc(edge_attr)
        for proc in self.processors:
            x, e = proc(x, edge_index, e)
        return self.flow_dec(x), self.heat_dec(x), self.species_dec(x)

ckpt = torch.load(CKPT, map_location='cpu')
cfg  = ckpt['cfg']
norm = ckpt['norm']
model = MultiHeadMGN(
    node_dim=cfg['node_input_dim'], edge_dim=cfg['edge_input_dim'],
    flow_out=cfg['flow_out_dim'],   heat_out=cfg['heat_out_dim'],
    species_out=cfg['species_out_dim'],
    hidden=cfg['hidden_dim'],       n_layers=cfg['n_layers'],
).to(DEVICE)
model.load_state_dict(ckpt['model'])
model.eval()
print('Surrogate loaded:', sum(p.numel() for p in model.parameters()), 'parameters')

In [ ]:
# ── Cell 7: Surrogate vs CFD ground-truth comparison ───────────────────────
node_mean = np.array(norm['node_mean'], dtype=np.float32)
node_std  = np.array(norm['node_std'],  dtype=np.float32)
out_mean  = np.array(norm['out_mean'],  dtype=np.float32)
out_std   = np.array(norm['out_std'],   dtype=np.float32)
FIELD_NAMES = ['Ux', 'Uy', 'Uz', 'p', 'T [K]', 'TMA']
K = cfg['k_neighbors']

def run_inference_h5(h5_path):
    with h5py.File(h5_path, 'r') as f:
        coords = f['coords'][:].astype(np.float32)
        nf     = f['inputs/node_features'][:].astype(np.float32)
        gf     = f['inputs/global'][:].astype(np.float32)
        truth  = f['outputs/node_fields'][:].astype(np.float32)  # [N, 6]
    N  = len(coords)
    xi = np.concatenate([nf, np.tile(gf, (N, 1))], axis=1)
    tree = cKDTree(coords)
    _, idx = tree.query(coords, k=K+1); idx = idx[:, 1:]
    src = np.repeat(np.arange(N), K); dst = idx.flatten()
    diff = coords[dst] - coords[src]
    dist = np.linalg.norm(diff, axis=1, keepdims=True)
    med  = float(np.median(dist)) + 1e-8
    ef   = np.concatenate([diff/med, dist/med], axis=1).astype(np.float32)
    x  = torch.from_numpy((xi - node_mean) / node_std).to(DEVICE)
    ei = torch.tensor(np.stack([src, dst]), dtype=torch.long).to(DEVICE)
    ea = torch.from_numpy(ef).to(DEVICE)
    with torch.no_grad():
        fp, hp, sp = model(x, ei, ea)
    fp = fp.cpu().numpy(); hp = hp.cpu().numpy().flatten(); sp = sp.cpu().numpy().flatten()
    del x, ei, ea
    preds = np.zeros((N, 6), dtype=np.float32)
    preds[:, :4] = fp * out_std[:4] + out_mean[:4]
    preds[:, 4]  = hp * out_std[4]  + out_mean[4]
    preds[:, 5]  = sp * out_std[5]  + out_mean[5]
    return coords, preds, truth, dict(zip(GLOBAL_KEYS, gf))

def ui_metric(field, z, frac_lo=0.10, frac_hi=0.55):
    """Uniformity index 1 - CV for a z-band."""
    z_lo = z.min() + frac_lo * (z.max() - z.min())
    z_hi = z.min() + frac_hi * (z.max() - z.min())
    m    = (z >= z_lo) & (z <= z_hi)
    if m.sum() < 10: return float('nan')
    cv = field[m].std() / (abs(field[m].mean()) + 1e-12)
    return float(1.0 - cv)

val_records = []

for d, (case_dir, h5_path) in zip(top3, postprocessed):
    if not h5_path.exists():
        print(f'[SKIP] Rank {d["rank"]}: HDF5 not found — run postprocess first'); continue

    print(f'\n=== Rank {d["rank"]} — {case_dir.name} ===')
    coords, preds, truth, gd = run_inference_h5(h5_path)
    z = coords[:, 2]

    # Error metrics
    errors = {}
    for i, fname in enumerate(FIELD_NAMES):
        rmse = float(np.sqrt(np.mean((preds[:, i] - truth[:, i])**2)))
        mae  = float(np.mean(np.abs(preds[:, i] - truth[:, i])))
        errors[fname] = {'rmse': rmse, 'mae': mae}
        print(f'  {fname:<8} RMSE={rmse:.4g}  MAE={mae:.4g}')

    # Uniformity comparison
    T_UI_cfd   = ui_metric(truth[:, 4], z, 0.0, 0.10)
    T_UI_surr  = ui_metric(preds[:, 4], z, 0.0, 0.10)
    TMA_UI_cfd  = ui_metric(truth[:, 5], z, 0.15, 0.55)
    TMA_UI_surr = ui_metric(preds[:, 5], z, 0.15, 0.55)
    print(f'  T_UI   CFD={T_UI_cfd:.4f}  surrogate={T_UI_surr:.4f}  '
          f'delta={T_UI_surr - T_UI_cfd:+.4f}')
    print(f'  TMA_UI CFD={TMA_UI_cfd:.4f}  surrogate={TMA_UI_surr:.4f}  '
          f'delta={TMA_UI_surr - TMA_UI_cfd:+.4f}')

    # Scatter: predicted vs truth for T and TMA
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, (fi, fname, unit) in zip(axes, [(4, 'T', 'K'), (5, 'TMA', '')]):
        ax.scatter(truth[:, fi][::20], preds[:, fi][::20], s=1, alpha=0.3)
        lo = min(truth[:, fi].min(), preds[:, fi].min())
        hi = max(truth[:, fi].max(), preds[:, fi].max())
        ax.plot([lo, hi], [lo, hi], 'r--', lw=1.5, label='y=x')
        ax.set_xlabel(f'CFD truth  {fname} [{unit}]')
        ax.set_ylabel(f'Surrogate  {fname} [{unit}]')
        ax.set_title(f'Rank {d["rank"]} — {fname}  RMSE={errors[fname+" [K]" if unit=="K" else fname]["rmse"]:.4g}')
        ax.legend(fontsize=8)
    fig.suptitle(f'Rank {d["rank"]}  D={d["D_mm"]}mm  pitch/D={d["pitch_over_D"]}  Q={d["Q_slm"]}slm')
    plt.tight_layout(); plt.show()

    # Field slice: CFD vs surrogate (bottom 15%)
    z_thresh = z.min() + 0.15 * (z.max() - z.min())
    m = z < z_thresh
    rng = np.random.default_rng(42)
    sel = rng.choice(m.sum(), min(3000, m.sum()), replace=False)
    xp, yp = coords[m][sel, 0]*1e3, coords[m][sel, 1]*1e3

    fig, axes = plt.subplots(2, 4, figsize=(16, 7))
    field_show = [(4,'T [K]'), (5,'TMA'), (3,'p'), (None,'|U|')]
    for col, (fi, fname) in enumerate(field_show):
        for row, (vals, label) in enumerate([
            (truth, 'CFD truth'), (preds, 'Surrogate')
        ]):
            fv = (np.linalg.norm(vals[m][sel, :3], axis=1)
                  if fi is None else vals[m][sel, fi])
            sc = axes[row, col].scatter(xp, yp, c=fv, s=1, cmap='turbo')
            plt.colorbar(sc, ax=axes[row, col], pad=0.02)
            axes[row, col].set_title(f'{label} — {fname}', fontsize=9)
            axes[row, col].set_aspect('equal'); axes[row, col].axis('off')
    fig.suptitle(f'Field slices (bottom 15% plenum) — Rank {d["rank"]}')
    plt.tight_layout(); plt.show()

    val_records.append({
        'rank':          d['rank'],
        'D_mm':          d['D_mm'],
        'pitch_over_D':  d['pitch_over_D'],
        'Q_slm':         d['Q_slm'],
        'h5_path':       str(h5_path),
        'surrogate_errors': {k: v for k, v in errors.items()},
        'T_UI_cfd':      T_UI_cfd,
        'T_UI_surr':     T_UI_surr,
        'TMA_UI_cfd':    TMA_UI_cfd,
        'TMA_UI_surr':   TMA_UI_surr,
    })

print(f'\nComparison complete for {len(val_records)} designs.')

In [ ]:
# ── Cell 8: Save validation_results.json ──────────────────────────────────
out = {
    'n_cfd_cases':   len(val_records),
    'cfd_val_cases': val_records,
}
with open(VAL_JSON, 'w') as f:
    json.dump(out, f, indent=2)
print(f'Saved → {VAL_JSON}')

# Summary table
print('\n  Rank  T_rmse      TMA_rmse    T_UI_err   TMA_UI_err')
print('  ----  ----------  ----------  ---------  ----------')
for r in val_records:
    T_err   = r['T_UI_surr']   - r['T_UI_cfd']
    TMA_err = r['TMA_UI_surr'] - r['TMA_UI_cfd']
    T_rmse  = r['surrogate_errors'].get('T [K]', {}).get('rmse', float('nan'))
    TMA_rmse = r['surrogate_errors'].get('TMA', {}).get('rmse', float('nan'))
    print(f"  {r['rank']:<6} {T_rmse:<12.4g} {TMA_rmse:<12.4g} {T_err:+.4f}    {TMA_err:+.4f}")

print('\nReady for 11_finetune_on_error.ipynb')